In [ ]:
import requests
from bs4 import BeautifulSoup
 
import time

In [ ]:
def get_clean_report_text(s):
    full_text = s.get_text(separator="\n")
    
    start_markers = [
        "HISTORY OF PRESENT ILLNESS",
        "CHIEF COMPLAINT",
        "REASON FOR VISIT",
        "SUBJECTIVE",
        "EMERGENCY DEPARTMENT",
        "PRESENTING COMPLAINT",
    ]
    end_markers = [
        "About This Sample",
        "Go Back to Emergency Room",
        "Related Samples",
        "Keywords:",
        "Educational Disclaimer",
    ]
    
    start_idx = None
    for marker in start_markers:
        idx = full_text.find(marker)
        if idx != -1:
            if start_idx is None or idx < start_idx:
                start_idx = idx

    end_idx = None
    for marker in end_markers:
        idx = full_text.find(marker)
        if idx != -1:
            if end_idx is None or idx < end_idx:
                end_idx = idx

    if start_idx is None:
        return None

    if end_idx is None or end_idx < start_idx:
        return full_text[start_idx:].strip()

    return full_text[start_idx:end_idx].strip()

In [ ]:
# ── Option 2: scrape ER reports directly (more efficient) ────────────────────


def get_er_samples():
    i=0
    records = []
    url = "https://www.mtsamples.com/site/pages/browse.asp?type=93-Emergency+Room+Reports"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a", href=True)
    sample_links = [
        l["href"] for l in links
        if "sample.asp" in l["href"] and "Emergency" in l["href"]
    ]
    print(f"Found {len(sample_links)} ER sample links")

    for link in sample_links:
        r = requests.get(link)
        s = BeautifulSoup(r.text, "html.parser")
        try:
            # Extract clean title from the Sample parameter in the URL
            sample_param = link.split("Sample=")[-1]
            title = sample_param.split("-", 1)[-1].replace("+", " ").replace("%2F", "/")

            text = get_clean_report_text(s)
            records.append({"title": title, "text": text})
            print(f"OK: {title}")
            i+=1
            print(i)
        except Exception as e:
            print(f"Skipping {link}: {e}")
        time.sleep(0.5)

    return pd.DataFrame(records)

In [ ]:
# ── 2. Load and explore ───────────────────────────────────────────────────────
df_er = get_er_samples()
print(f"\nTotal ER reports collected: {len(df_er)}")

### Save data in a folder to explore

In [ ]:
import os
import json
import re

# ── 1. Save as CSV ────────────────────────────────────────────────────────────
output_dir = "mtsamples_er"
os.makedirs(output_dir, exist_ok=True)

# Save the full dataframe as CSV
df_er.to_csv(os.path.join(output_dir, "er_reports.csv"), index=False)
print(f"Saved {len(df_er)} reports to {output_dir}/er_reports.csv")

# ── 2. Optionally save each report as its own .txt file ──────────────────────
txt_dir = os.path.join(output_dir, "texts")
os.makedirs(txt_dir, exist_ok=True)

for _, row in df_er.iterrows():
    # Clean the title: strip whitespace/newlines, then sanitize for filename
    clean_title = " ".join(row['title'].split())  # collapses all whitespace and newlines
    clean_title = re.sub(r'Sample Name:\s*', '', clean_title)  # remove "Sample Name:" prefix
    clean_title = re.sub(r'Medical Specialty:.*', '', clean_title).strip()  # remove everything from "Medical Specialty:" onward
    clean_title = re.sub(r'[^\w\s-]', '', clean_title)  # remove special chars like &
    filename = clean_title.replace(" ", "_") + ".txt"
    
    filepath = os.path.join(txt_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(row['text'] or "")

print(f"Saved individual .txt files to {txt_dir}/")

# Verify the filenames look clean
import os
for f in sorted(os.listdir(txt_dir))[:5]:
    print(f)





In [ ]:
# ── 3. Explore ────────────────────────────────────────────────────────────────
# Basic stats
print(f"\nTotal reports:   {len(df_er)}")
print(f"Missing texts:   {df_er['text'].isna().sum()}")
print(f"Avg text length: {df_er['text'].str.len().mean():.0f} chars")

# Look at a random example
sample = df_er.sample(1).iloc[0]
print(f"\n── Random example ──────────────────────────────")
print(f"Title: {sample['title']}")
print(f"Text:\n{sample['text'][:5000]}...")